# Unzip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/Individual Honour Project/code/dataset/final_dataset_dat_test.zip"
extract_path = "/content/dataset/final_dataset_test"

os.makedirs(extract_path, exist_ok=True)

In [ ]:
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extraction completed directly in Google Drive")

# Setup

In [ ]:
!pip install kornia

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from collections import Counter
from typing import Dict, List, Optional, Tuple
from copy import deepcopy
from torch.optim.swa_utils import AveragedModel, update_bn

import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, roc_auc_score, confusion_matrix, roc_curve
)

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import pandas as pd
import kornia as K
import torch.nn.functional as F
import math
from collections import Counter

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset

class MemmapImageDataset(Dataset):
    def __init__(self, X, y, meta_path):
        """
        X: np.memmap of images (N, 3, H, W)
        y: np.memmap or np.ndarray of labels (N,)
        meta_path: path to meta.npy
        """
        self.X = X
        self.y = y

        # ---- Load metadata ----
        meta = np.load(meta_path, allow_pickle=True).item()
        class_map = meta["class_map"]

        # Ensure correct index order
        self.classes = [None] * len(class_map)
        for class_name, idx in class_map.items():
            self.classes[idx] = class_name

        self.class_to_idx = class_map

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # clone() is REQUIRED for memmap safety
        x = torch.from_numpy(self.X[idx]).float().clone()
        y = int(self.y[idx])
        return x, y


In [ ]:
# DATASET_DIR = "./dataset/cas_processed"   # your dataset folder
DATASET_DIR = "/content/dataset/final_dataset_test"   # your dataset folder
BATCH_SIZE = 32
IMAGE_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_MODEL_DIR = "/content/drive/MyDrive/Individual Honour Project/code/models"

In [ ]:
meta = np.load(f"{DATASET_DIR}/meta_test.npy", allow_pickle=True).item()

N_SAMPLES = meta["shape"][0]          # total samples
IMG_SHAPE = meta["shape"][1:]          # (3,224,224)

In [ ]:
y = np.memmap(
    f"{DATASET_DIR}/y_test.dat",
    dtype=np.int64,
    mode="r",
    shape=(N_SAMPLES,)
)

# N_TEST = len(y)

X = np.memmap(
    f"{DATASET_DIR}/X_test.dat",
    dtype=np.float16,
    mode="r",
    shape=(N_SAMPLES,3,224,224)
)


In [ ]:
dataset = MemmapImageDataset(
    X=X,
    y=y,
    meta_path=f"{DATASET_DIR}/meta_test.npy"
)
class_names = dataset.classes
NUM_CLASSES = len(class_names)
print(class_names)

In [ ]:
class StackedEnsemble(nn.Module):
    def __init__(self, num_classes=3, v2=False):
        super().__init__()
        self.v2 = v2
        # Backbone models
        self.effnet = load_architecture("efficientnet_b3")
        self.mobilenet = load_architecture("mobilenet")
        if v2:
            self.convnext = load_architecture("convnext_single")
        else:
            self.resnet = load_architecture("resnet18")

        # Meta classifier (9 → hidden → 3)
        self.meta = nn.Sequential(
            nn.Linear(9, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, num_classes)
        )

    def forward(self, x):

        # Backbone predictions
        p1 = torch.softmax(self.effnet(x), dim=1)
        p3 = torch.softmax(self.mobilenet(x), dim=1)
        if self.v2:
            p2 = torch.softmax(self.convnext(x), dim=1)
        else:
            p2 = torch.softmax(self.resnet(x), dim=1)

        # Concatenate → 9 feature vector
        stacked = torch.cat([p1, p2, p3], dim=1)

        # Meta classifier
        out = self.meta(stacked)

        return out

In [ ]:
class StackedEnsembleLarge(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        # Backbone models
        self.effnet = load_architecture("efficientnet_b5")
        self.mobilenet = load_architecture("mobilenet_large")
        self.resnet = load_architecture("resnet50")

        # Meta classifier (9 → hidden → 3)
        input_dim = num_classes * 3

        self.meta = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )


    def forward(self, x):

        # Backbone predictions
        p1 = torch.softmax(self.effnet(x), dim=1)
        p3 = torch.softmax(self.mobilenet(x), dim=1)
        p2 = torch.softmax(self.resnet(x), dim=1)

        # Concatenate → 9 feature vector
        stacked = torch.cat([p1, p2, p3], dim=1)

        # Meta classifier
        out = self.meta(stacked)

        return out


In [ ]:
class EnsembleTransformer(nn.Module):
    def __init__(self, num_classes=3, dim=64, depth=2, heads=4):
        super().__init__()

        self.embed = nn.Linear(num_classes, dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=dim * 2,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth
        )

        self.norm = nn.LayerNorm(dim)

        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        """
        x: (B, 3 models, num_classes=3)
        """

        x = self.embed(x)              # (B, 3, dim)
        x = self.transformer(x)        # (B, 3, dim)

        x = x.mean(dim=1)              # global token pooling
        x = self.norm(x)

        return self.head(x)

class StackedTransformerEnsemble(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.effnet = load_architecture("efficientnet_b3")
        self.resnet = load_architecture("resnet18")
        self.mobilenet = load_architecture("mobilenet")

        # freeze them
        for m in [self.effnet, self.resnet, self.mobilenet]:
            for p in m.parameters():
                p.requires_grad = False

        self.meta = EnsembleTransformer(num_classes)

    def forward(self, x):

        logits1 = self.effnet(x)
        logits2 = self.resnet(x)
        logits3 = self.mobilenet(x)

        # stack tokens
        tokens = torch.stack([logits1, logits2, logits3], dim=1)
        # (B, 3, 3)

        return self.meta(tokens)

In [ ]:
class WeightedEnsemble(nn.Module):
    def __init__(self, num_classes=3, v2=False):
        super().__init__()

        self.v2 = v2
        self.num_classes = num_classes
        self.eps = 1e-6

        embed_dim = 96   # 🔥 increased capacity

        # ===== Backbones =====
        self.effnet = load_architecture("efficientnet_b3")
        self.mobilenet = load_architecture("mobilenet")

        if v2:
            self.convnext = load_architecture("convnext_single")
        else:
            self.resnet = load_architecture("resnet18")

        # ===============================
        # 🔥 PER-MODEL TEMPERATURE (big gain)
        # ===============================
        self.temperature = nn.Parameter(torch.ones(3) * 1.5)

        # ===============================
        # 🔥 CROSS ATTENTION (stronger)
        # ===============================
        self.token_proj = nn.Linear(num_classes, embed_dim)

        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=4,
            dropout=0.1,
            batch_first=True
        )

        # 🔥 learned token importance (better than mean)
        self.token_weight = nn.Linear(embed_dim, 1)

        self.out_proj = nn.Linear(embed_dim, num_classes)

        # ===============================
        # 🔥 GLOBAL PRIOR
        # ===============================
        self.logit_weights = nn.Parameter(torch.ones(3))

        # 🔥 learnable fusion ratio
        self.alpha = nn.Parameter(torch.tensor(0.5))

        # ===============================
        # 🔥 STRONGER HEAD
        # ===============================
        self.head = nn.Sequential(
            nn.LayerNorm(num_classes),
            nn.Linear(num_classes, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def safe_logits(self, x):
        x = x - x.mean(dim=1, keepdim=True)
        x = x / (x.std(dim=1, keepdim=True) + 1e-6)
        return torch.tanh(x) * 5.0

    def forward(self, x):

        # ===== logits =====
        l1 = self.safe_logits(self.effnet(x))
        l3 = self.safe_logits(self.mobilenet(x))
        l2 = self.safe_logits(self.convnext(x) if self.v2 else self.resnet(x))

        # ===== per-model temperature =====
        T = torch.clamp(self.temperature, 0.5, 3.0)
        l1, l2, l3 = l1 / T[0], l2 / T[1], l3 / T[2]

        # ===== Dirichlet =====
        def evidence(logits):
            return F.softplus(logits)

        a1, a2, a3 = evidence(l1)+1, evidence(l2)+1, evidence(l3)+1

        p1 = a1 / (a1.sum(dim=1, keepdim=True) + self.eps)
        p2 = a2 / (a2.sum(dim=1, keepdim=True) + self.eps)
        p3 = a3 / (a3.sum(dim=1, keepdim=True) + self.eps)

        # ===============================
        # 🔥 CROSS ATTENTION (IMPROVED)
        # ===============================
        tokens = torch.stack([p1, p2, p3], dim=1)   # (B,3,C)
        tokens = self.token_proj(tokens)            # (B,3,D)

        attn_out, _ = self.attn(tokens, tokens, tokens)

        # 🔥 learned pooling instead of mean
        weights = torch.softmax(self.token_weight(attn_out), dim=1)  # (B,3,1)
        fused_attn = (weights * attn_out).sum(dim=1)  # (B,D)

        fused_attn = self.out_proj(fused_attn)

        # ===============================
        # 🔥 GLOBAL PRIOR
        # ===============================
        w = torch.softmax(self.logit_weights, dim=0)
        global_out = w[0]*p1 + w[1]*p2 + w[2]*p3

        # ===============================
        # 🔥 LEARNABLE FUSION
        # ===============================
        alpha = torch.sigmoid(self.alpha)
        fused = alpha * fused_attn + (1 - alpha) * global_out

        # ===============================
        out = self.head(fused)
        return out


In [ ]:
class BoostedTransformer(nn.Module):
    def __init__(self, num_classes=3, embed_dim=128):
        super().__init__()

        # ===== Base models =====
        self.m1 = load_architecture("efficientnet_b3")
        self.m2 = load_architecture("mobilenet")
        self.m3 = load_architecture("resnet18")

        # Freeze early layers, allow last layers to adapt
        for m in [self.m1, self.m2, self.m3]:
            for name, p in m.named_parameters():
                if "layer4" not in name and "head" not in name:
                    p.requires_grad = False

        # ===== Residual learners =====
        self.res1 = nn.Linear(num_classes, num_classes)
        self.res2 = nn.Linear(num_classes, num_classes)

        # ===== Token projection =====
        self.token_proj = nn.Linear(num_classes, embed_dim)

        # ===== Transformer encoder =====
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=4,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # ===== Attention pooling =====
        self.pool = nn.Linear(embed_dim, 1)

        # ===== Final head =====
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # ===== Stage 1 =====
        l1 = self.m1(x)

        # ===== Stage 2 (residual learning) =====
        l2 = self.m2(x)
        l2 = l2 + self.res1(l1)

        # ===== Stage 3 =====
        l3 = self.m3(x)
        l3 = l3 + self.res2(l2)

        # ===== Stack tokens =====
        tokens = torch.stack([l1, l2, l3], dim=1)  # (B,3,C)

        tokens = self.token_proj(tokens)

        # ===== Transformer =====
        tokens = self.transformer(tokens)

        # ===== Attention pooling =====
        weights = torch.softmax(self.pool(tokens), dim=1)
        fused = (weights * tokens).sum(dim=1)

        return self.head(fused)


In [ ]:
class ConvBNAct(nn.Module):
    """Conv2d → BN → SiLU (or custom act).  padding auto-computed if omitted."""

    def __init__(
        self,
        in_ch : int,
        out_ch: int,
        k     : int                 = 3,
        stride: int                 = 1,
        groups: int                 = 1,
        dil   : int                 = 1,
        bias  : bool                = False,
        act   : Optional[nn.Module] = None,
    ) -> None:
        super().__init__()
        pad        = (k - 1) // 2 * dil
        self.conv  = nn.Conv2d(in_ch, out_ch, k, stride=stride, padding=pad,
                               dilation=dil, groups=groups, bias=bias)
        self.bn    = nn.BatchNorm2d(out_ch, momentum=0.01, eps=1e-3)
        self.act   = act if act is not None else nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))

class MultiScaleEdgeExtractor(nn.Module):
    """
    Fixed gradient-operator bank (7 channels) followed by a learnable
    depthwise-separable refinement head.

    Operator bank (each map normalised per-sample to [0, 1]):
        ch 0   Sobel magnitude     ||grad I||_2
        ch 1   |Sobel Ix|          horizontal edge strength
        ch 2   |Sobel Iy|          vertical edge strength
        ch 3   |Laplacian 5×5|     second-order isotropic response
        ch 4   |Scharr Ix|         rotation-invariant horizontal gradient
        ch 5   DoG σ=(1.0, 2.0)    fine-scale blob / edge
        ch 6   DoG σ=(2.0, 4.0)    coarse-scale boundary

    The fixed kernels are buffers (not parameters) so they follow
    .to(device) / .half() calls automatically.

    Refinement path:
        3×3 ConvBNAct   mix all 7 responses
        DW 3×3 ConvBNAct  spatial refinement (zero cross-channel cost)
        PW 1×1 ConvBNAct  channel mixing / dimensionality control

    Args:
        out_ch : refined output channels  (default 16)
    """

    _DOG_PAIRS: Tuple[Tuple[float, float], ...] = ((1.0, 2.0), (2.0, 4.0))

    def __init__(self, out_ch: int = 16) -> None:
        super().__init__()
        self.register_buffer(
            "sobel_x",
            torch.tensor([[-1., 0., 1.],
                          [-2., 0., 2.],
                          [-1., 0., 1.]]).view(1, 1, 3, 3),
        )
        self.register_buffer(
            "sobel_y",
            torch.tensor([[-1., -2., -1.],
                          [ 0.,  0.,  0.],
                          [ 1.,  2.,  1.]]).view(1, 1, 3, 3),
        )
        self.register_buffer(
            "scharr_x",
            torch.tensor([[ -3., 0.,  3.],
                          [-10., 0., 10.],
                          [ -3., 0.,  3.]]).view(1, 1, 3, 3),
        )

        self.refine = nn.Sequential(
            ConvBNAct(7, out_ch, k=3),
            ConvBNAct(out_ch, out_ch, k=3, groups=out_ch),  # depthwise
            ConvBNAct(out_ch, out_ch, k=1),                  # pointwise
        )

    @staticmethod
    def _minmax_norm(t: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
        B, C, H, W = t.shape
        flat = t.reshape(B, C, -1)
        lo   = flat.min(dim=-1).values.view(B, C, 1, 1)
        hi   = flat.max(dim=-1).values.view(B, C, 1, 1)
        return (t - lo) / (hi - lo + eps)

    def _dog(self, gray: torch.Tensor) -> torch.Tensor:
        out: List[torch.Tensor] = []
        for s1, s2 in self._DOG_PAIRS:
            ks1 = max(3, 2 * math.ceil(2 * s1) + 1) | 1
            ks2 = max(3, 2 * math.ceil(2 * s2) + 1) | 1
            out.append(
                K.filters.gaussian_blur2d(gray, (ks1, ks1), (s1, s1))
                - K.filters.gaussian_blur2d(gray, (ks2, ks2), (s2, s2))
            )
        return torch.cat(out, dim=1)   # (B, 2, H, W)

    # AFTER — renamed to _conv2d to avoid collision
    def _conv2d(self, gray: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
        return F.conv2d(gray, kernel.to(dtype=gray.dtype), padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gray = K.color.rgb_to_grayscale(x)
        raw  = torch.cat([
            K.filters.sobel(gray),
            self._conv2d(gray, self.sobel_x).abs(),
            self._conv2d(gray, self.sobel_y).abs(),
            K.filters.laplacian(gray, kernel_size=5).abs(),
            self._conv2d(gray, self.scharr_x).abs(),
            self._dog(gray),
        ], dim=1)
        return self.refine(self._minmax_norm(raw))

class EdgeCrossAttention(nn.Module):
    """
    Spatial cross-attention: backbone features supply Q; edge maps supply K, V.

    This injects geometric structure cues (scarps, drainage channels,
    ridgelines) into backbone features without permanently widening the
    feature tensor.  A learnable scalar γ, initialised at 0, gates the
    module as a warm-up:
        output = feat + γ · cross_attn(feat, edge)
    so the module starts as an exact identity.

    Args:
        feat_ch : backbone feature channels (Q dimension)
        edge_ch : refined edge descriptor channels (K/V source)
        heads   : number of attention heads
        dropout : attention-weight dropout
    """

    def __init__(
        self,
        feat_ch: int,
        edge_ch: int,
        heads  : int   = 4,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        assert feat_ch % heads == 0
        self.heads    = heads
        self.head_dim = feat_ch // heads
        self.scale    = self.head_dim ** -0.5

        self.q_proj = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 1, bias=False),
            nn.GroupNorm(heads, feat_ch),
        )
        self.k_proj = nn.Conv2d(edge_ch, feat_ch, 1, bias=False)
        self.v_proj = nn.Conv2d(edge_ch, feat_ch, 1, bias=False)

        self.out_proj = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 1, bias=False),
            nn.BatchNorm2d(feat_ch, momentum=0.01, eps=1e-3),
        )
        self.drop  = nn.Dropout(p=dropout)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, feat: torch.Tensor, edge: torch.Tensor) -> torch.Tensor:
        B, C, H, W = feat.shape
        N = H * W

        if edge.shape[-2:] != (H, W):
            edge = F.interpolate(edge, (H, W), mode="bilinear", align_corners=False)

        def _heads(t: torch.Tensor) -> torch.Tensor:
            return t.view(B, self.heads, self.head_dim, N).permute(0, 1, 3, 2)

        Q  = _heads(self.q_proj(feat))
        K_ = _heads(self.k_proj(edge))
        V_ = _heads(self.v_proj(edge))

        w   = self.drop(F.softmax(torch.matmul(Q, K_.transpose(-2, -1)) * self.scale, dim=-1))
        out = torch.matmul(w, V_).permute(0, 1, 3, 2).contiguous().view(B, C, H, W)
        return feat + self.gamma * self.out_proj(out)

class EdgeAttentionModule(nn.Module):
    """
    Multi-Scale Edge Attention Module (MEAM).

    Composes MultiScaleEdgeExtractor + EdgeCrossAttention.
    The extractor always receives the original full-resolution image so that
    fine boundary detail is never lost to intermediate downsampling.

    Inserted after features[3] (C=48) because:
        - Mid-resolution (~28×28 for 224px) preserves boundary spatial coherence.
        - This is the FPN P3 tap so edge-enriched features feed the finest
          pyramid level directly.

    Args:
        feat_ch  : backbone feature channels at insertion point (48 for B3)
        edge_ch  : edge descriptor output channels
        heads    : cross-attention heads
        dropout  : attention dropout
    """

    def __init__(
        self,
        feat_ch: int,
        edge_ch: int   = 16,
        heads  : int   = 4,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.extractor  = MultiScaleEdgeExtractor(out_ch=edge_ch)
        self.cross_attn = EdgeCrossAttention(feat_ch, edge_ch, heads, dropout)
        self.post_norm  = nn.BatchNorm2d(feat_ch, momentum=0.01, eps=1e-3)

    def forward(self, feat: torch.Tensor, img: torch.Tensor) -> torch.Tensor:
        """
        feat : (B, C, H, W)    backbone feature at features[3] output
        img  : (B, 3, H0, W0)  original full-resolution input image
        """
        return self.post_norm(self.cross_attn(feat, self.extractor(img)))

class DilatedMultiScaleBranch(nn.Module):
    """
    Four parallel dilated depthwise-separable convolution branches with
    learnable per-branch softmax mixture weights.

    Dilation rates {1, 2, 4, 8} give exponentially growing effective
    receptive fields at identical per-branch parameter counts.  The outputs
    are combined as a learned convex combination and projected back to in_ch.

    Each branch:
        DW Conv2d (rate r, 3×3) → BN → SiLU → PW Conv2d (1×1)

    Args:
        in_ch  : input (and output) channel count
        mid_ch : internal width per branch
    """

    _RATES: Tuple[int, ...] = (1, 2, 4, 8)

    def __init__(self, in_ch: int, mid_ch: int) -> None:
        super().__init__()
        self.reduce = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 1, bias=False),
            nn.BatchNorm2d(mid_ch, momentum=0.01, eps=1e-3),
            nn.SiLU(inplace=True),
        )
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(mid_ch, mid_ch, 3,
                          padding=r, dilation=r, groups=mid_ch, bias=False),
                nn.BatchNorm2d(mid_ch, momentum=0.01, eps=1e-3),
                nn.SiLU(inplace=True),
                nn.Conv2d(mid_ch, mid_ch, 1, bias=False),
            )
            for r in self._RATES
        ])
        self.branch_weights = nn.Parameter(torch.ones(len(self._RATES)))
        self.expand = nn.Sequential(
            nn.Conv2d(mid_ch, in_ch, 1, bias=False),
            nn.BatchNorm2d(in_ch, momentum=0.01, eps=1e-3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.reduce(x)
        w = F.softmax(self.branch_weights, dim=0)
        out = sum(w_i * branch(h) for w_i, branch in zip(w.unbind(), self.branches))
        return self.expand(out)

class SCAM_Enhanced(nn.Module):
    """
    Enhanced Spatial-Channel Attention Module (SCAM+).

    Novel contributions over the original SCAM:
        1. SE-style channel excitation (avg + max → shared MLP → sigmoid)
           pre-weights the input before computing the combined descriptor Fsc.
        2. Dilated multi-scale branch (rates {1,2,4,8} DW-separable) replaces
           fixed-kernel multi-branch for broader receptive field coverage.
        3. Fast 1×1 shortcut branch concatenated with the dilated output for
           dual-path aggregation before the final projection.
        4. Sigmoid residual gate γ initialised at 0.1:
               output = x + x ⊙ attn ⊙ γ

    Inserted after features[5] (C=136) because:
        - Mid-semantic depth encodes landslide texture and shape patterns.
        - Dilated branches are most cost-effective at this intermediate depth.
        - This is the FPN P4 tap; calibrating features here improves the
          quality of the mid-level pyramid contribution.

    Args:
        in_ch     : input / output channel count
        reduction : channel excitation MLP bottleneck ratio
    """

    def __init__(self, in_ch: int, reduction: int = 16) -> None:
        super().__init__()
        mid = max(in_ch // reduction, 8)

        self.ch_mlp = nn.Sequential(
            nn.Linear(in_ch, mid),
            nn.GELU(),
            nn.Linear(mid, in_ch),
        )
        self.spatial_enc = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1, bias=False),
            nn.BatchNorm2d(mid, momentum=0.01, eps=1e-3),
            nn.SiLU(inplace=True),
        )
        self.dilated  = DilatedMultiScaleBranch(mid, max(mid // 2, 4))
        self.shortcut = nn.Sequential(
            nn.Conv2d(mid, mid, 1, bias=False),
            nn.BatchNorm2d(mid, momentum=0.01, eps=1e-3),
        )
        self.aggr = nn.Sequential(
            nn.Conv2d(mid * 2, in_ch, 1, bias=False),
            nn.BatchNorm2d(in_ch, momentum=0.01, eps=1e-3),
        )
        self.gate    = nn.Parameter(torch.tensor(0.1))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape

        avg_c   = F.adaptive_avg_pool2d(x, 1).view(B, C)
        max_c   = F.adaptive_max_pool2d(x, 1).view(B, C)
        ch_attn = self.sigmoid(self.ch_mlp(avg_c) + self.ch_mlp(max_c)).view(B, C, 1, 1)

        sp_bias = (x.mean(dim=1, keepdim=True) + x.max(dim=1, keepdim=True).values) * 0.5
        Fsc     = x * ch_attn * (1.0 + sp_bias)

        h    = self.spatial_enc(Fsc)
        attn = self.sigmoid(self.aggr(torch.cat([self.dilated(h), self.shortcut(h)], dim=1)))
        return x + x * attn * self.gate

class ChannelAttention3D(nn.Module):
    """
    Channel Attention with 3-D Convolution over a Spatial-Pyramid Descriptor.

    Standard CBAM-CA uses single-scale global pooling → MLP, discarding all
    spatial structure.  This module pools at two scales (1×1, 2×2) yielding
    5 descriptors per channel, packed into a 3-D volume (B, 1, C, 5, 1).
    A Conv3d with kernel (3,1,1) attends to inter-channel neighbourhoods
    jointly with the pooling-scale axis, a strictly richer excitation.

    Inserted after features[7] (C=384) because high-level abstract features
    benefit from pyramid-pooling context, and at this depth the spatial
    resolution (~7×7) is too low for dilated convolutions to be meaningful.

    Args:
        channels  : input channel count  (384 for B3 features[7])
        reduction : MLP bottleneck ratio
    """

    def __init__(self, channels: int, reduction: int = 16) -> None:
        super().__init__()
        mid = max(channels // reduction, 8)

        self.conv3d_avg = nn.Sequential(
            nn.Conv3d(1, 1, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False),
            nn.ReLU(inplace=True),
        )
        self.conv3d_max = nn.Sequential(
            nn.Conv3d(1, 1, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False),
            nn.ReLU(inplace=True),
        )
        self.mlp = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
        )
        self.sigmoid = nn.Sigmoid()

    def _pack(self, x: torch.Tensor, pool_fn) -> torch.Tensor:
        B, C = x.shape[:2]
        p1 = pool_fn(x, 1).view(B, C, 1)
        p2 = pool_fn(x, 2).view(B, C, 4)
        return torch.cat([p1, p2], dim=2).unsqueeze(1).unsqueeze(-1)  # (B,1,C,5,1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        avg_vol = self._pack(x, F.adaptive_avg_pool2d)
        max_vol = self._pack(x, F.adaptive_max_pool2d)

        avg_out = self.conv3d_avg(avg_vol).squeeze(1).squeeze(-1)  # (B,C,5)
        max_out = self.conv3d_max(max_vol).squeeze(1).squeeze(-1)

        ca = self.sigmoid(
            self.mlp(avg_out.mean(dim=-1)) + self.mlp(max_out.mean(dim=-1))
        )
        return x * ca.view(B, C, 1, 1)

class SpatialAttention3D(nn.Module):
    """
    Spatial Attention with 3-D Convolution over a Four-Statistic Volume.

    Standard CBAM-SA stacks {mean, max} → (B, 2, H, W) and uses a 2-D conv,
    losing inter-statistic relationships.  This module computes four statistics
    {mean, max, std, min} reshaped as a 3-D volume (B, 1, 4, H, W).  A Conv3d
    with depth kernel D=3 jointly models inter-statistic correlations and
    spatial context.  A second Conv3d (D=4) collapses the statistic axis with
    learned weights (not a fixed mean).

    Args:
        kernel_size : 2-D spatial kernel of the joint Conv3d (default 7)
    """

    def __init__(self, kernel_size: int = 7) -> None:
        super().__init__()
        pad = kernel_size // 2
        self.conv3d_joint = nn.Sequential(
            nn.Conv3d(1, 1, kernel_size=(3, kernel_size, kernel_size),
                      padding=(1, pad, pad), bias=False),
            nn.BatchNorm3d(1, momentum=0.01, eps=1e-3),
            nn.ReLU(inplace=True),
        )
        self.stat_collapse = nn.Conv3d(1, 1, kernel_size=(4, 1, 1), bias=False)
        self.sigmoid        = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean_s   = x.mean(dim=1, keepdim=True)
        max_s, _ = x.max(dim=1, keepdim=True)
        std_s    = x.std(dim=1, keepdim=True)
        min_s, _ = x.min(dim=1, keepdim=True)

        vol = torch.cat([mean_s, max_s, std_s, min_s], dim=1).unsqueeze(1)  # (B,1,4,H,W)
        out = self.stat_collapse(self.conv3d_joint(vol))                      # (B,1,1,H,W)
        return x * self.sigmoid(out.squeeze(2))

class CBAM3D(nn.Module):
    """
    3-D Convolutional Bottleneck Attention Module.

    Applies ChannelAttention3D → SpatialAttention3D sequentially, wrapped in
    a soft residual gate initialised at 0 (module is an identity at the start
    of training and gradually activates):
        output = x + sigmoid(gate) * (CBAM(x) - x)

    Inserted after features[7] (C=384) where high-level semantic features
    benefit from pyramid-pooling attention.

    Args:
        channels       : input / output channel count
        reduction      : channel MLP bottleneck ratio
        spatial_kernel : spatial kernel size of the 3-D spatial attention
    """

    def __init__(
        self,
        channels      : int = 384,
        reduction     : int = 16,
        spatial_kernel: int = 7,
    ) -> None:
        super().__init__()
        self.channel_attn = ChannelAttention3D(channels, reduction)
        self.spatial_attn = SpatialAttention3D(spatial_kernel)
        self.gate         = nn.Parameter(torch.zeros(1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attended = self.spatial_attn(self.channel_attn(x))
        return x + torch.sigmoid(self.gate) * (attended - x)

class LateralFusionBlock(nn.Module):
    """
    One FPN lateral connection: coarser top-down map fused with a finer
    lateral map via learnable α-blending.

    fused = sigmoid(α) * proj(upsample(top)) + (1-sigmoid(α)) * proj(lat)

    A 3×3 ConvBNAct smooths the fused output.

    Args:
        top_ch : coarser input channels
        lat_ch : finer input channels
        out_ch : unified output channels
    """

    def __init__(self, top_ch: int, lat_ch: int, out_ch: int) -> None:
        super().__init__()
        self.top_proj = nn.Sequential(
            nn.Conv2d(top_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch, momentum=0.01, eps=1e-3),
        )
        self.lat_proj = nn.Sequential(
            nn.Conv2d(lat_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch, momentum=0.01, eps=1e-3),
        )
        self.smooth = ConvBNAct(out_ch, out_ch, k=3)
        self.alpha  = nn.Parameter(torch.tensor(0.5))

    def forward(self, top: torch.Tensor, lat: torch.Tensor) -> torch.Tensor:
        top_up  = F.interpolate(self.top_proj(top), size=lat.shape[-2:],
                                mode="bilinear", align_corners=False)
        a       = torch.sigmoid(self.alpha)
        return self.smooth(a * top_up + (1.0 - a) * self.lat_proj(lat))

class MultiScaleFeatureFusion(nn.Module):
    """
    3-level top-down FPN: P5 (coarse) → P4 → P3 (fine).

    The fused P3-resolution output is refined by an additional CBAM3D to
    suppress upsampling artefacts introduced by the top-down path.

    EfficientNet-B3 channel widths at each tap:
        P3: 48   (after features[3] + EdgeAttn)
        P4: 136  (after features[5] + SCAM+)
        P5: 1536 (after features[8], top conv)

    Args:
        ch_p3  : P3 channel count
        ch_p4  : P4 channel count
        ch_p5  : P5 channel count
        out_ch : unified FPN output channels
    """

    def __init__(
        self,
        ch_p3 : int = 48,
        ch_p4 : int = 136,
        ch_p5 : int = 1536,
        out_ch: int = 256,
    ) -> None:
        super().__init__()
        self.p5_to_p4 = LateralFusionBlock(ch_p5, ch_p4, out_ch)
        self.p4_to_p3 = LateralFusionBlock(out_ch, ch_p3, out_ch)
        self.refine   = nn.Sequential(
            CBAM3D(out_ch, reduction=16, spatial_kernel=7),
            ConvBNAct(out_ch, out_ch, k=1),
        )

    def forward(self, p3: torch.Tensor, p4: torch.Tensor, p5: torch.Tensor) -> torch.Tensor:
        m4 = self.p5_to_p4(p5, p4)
        m3 = self.p4_to_p3(m4, p3)
        return self.refine(m3)

class ClassificationHead(nn.Module):
    """
    Multi-scale pool (1×1 + 2×2) → LayerNorm → 3-layer GELU MLP.

    Pools the FPN output at two spatial scales:
        1×1 global average → (B, C)
        2×2 average        → (B, 4C)   four spatial quadrant descriptors
    Concatenated to (B, 5C), normalised by LayerNorm, then MLP → logits.

    Args:
        in_ch       : FPN output channel count
        num_classes : number of output classes
        dropout     : dropout after the first two linear layers
    """

    def __init__(self, in_ch: int, num_classes: int, dropout: float = 0.3) -> None:
        super().__init__()
        feat_dim = in_ch * 5
        self.norm = nn.LayerNorm(feat_dim)
        self.mlp  = nn.Sequential(
            nn.Linear(feat_dim, in_ch),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(in_ch, in_ch // 2),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(in_ch // 2, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        p1 = F.adaptive_avg_pool2d(x, 1).flatten(1)
        p2 = F.adaptive_avg_pool2d(x, 2).flatten(1)
        return self.mlp(self.norm(torch.cat([p1, p2], dim=1)))

class LandslideEfficientNet(nn.Module):
    """
    Pretrained EfficientNet-B3 backbone with novel attention modules inserted
    at three principled tap points, fused by a lightweight FPN.

    Forward flow:
        img
         │
         ├─ features[0..3] ──► EdgeAttentionModule(feat, img) ──► P3 (C=48)
         │                                                          │
         ├─ features[4..5] ──► SCAM_Enhanced ──────────────────► P4 (C=136)
         │                                                          │
         ├─ features[6..7] ──► CBAM3D ──► features[8] ──────────► P5 (C=1536)
         │                                                          │
         └────────────────────────────────────── FPN(P3,P4,P5) ──► ClassificationHead
                                                                    │
                                                                 logits (B, num_classes)

    Channel widths at tap points are fixed by EfficientNet-B3:
        P3: 48  P4: 136  P5: 1536

    Args:
        num_classes : number of output classes  (default 3)
        fpn_ch      : unified FPN output channels  (default 256)
        dropout     : classification head dropout  (default 0.3)
        pretrained  : load ImageNet-1K weights for the backbone  (default True)
    """

    # EfficientNet-B3 output channels at each stage boundary
    _CH_P3:  int = 48     # features[3] output
    _CH_P4:  int = 136    # features[5] output
    _CH_PRE_TOP: int = 384    # features[7] output  (CBAM3D applied here)
    _CH_P5:  int = 1536   # features[8] output  (top conv)

    def __init__(
        self,
        num_classes: int  = 3,
        fpn_ch     : int  = 256,
        dropout    : float = 0.3,
        pretrained : bool  = True,
    ) -> None:
        super().__init__()

        weights = "IMAGENET1K_V1" if pretrained else None
        backbone = models.efficientnet_b3(weights=weights)

        # features[0..8]; we split them for manual stage-wise execution
        self.f0_3 = nn.Sequential(
            backbone.features[0],
            backbone.features[1],
            backbone.features[2],
            backbone.features[3],
        )
        self.f4_5 = nn.Sequential(
            backbone.features[4],
            backbone.features[5],
        )
        self.f6_7 = nn.Sequential(
            backbone.features[6],
            backbone.features[7],
        )
        self.f8 = backbone.features[8]   # top conv, out=1536

        self.edge_attn = EdgeAttentionModule(
            feat_ch = self._CH_P3,
            edge_ch = 16,
            heads   = 4,           # 48 / 4 = 12 head_dim
            dropout = 0.1,
        )

        self.scam = SCAM_Enhanced(
            in_ch     = self._CH_P4,
            reduction = 16,
        )

        self.cbam3d = CBAM3D(
            channels       = self._CH_PRE_TOP,
            reduction      = 16,
            spatial_kernel = 7,
        )

        self.fpn  = MultiScaleFeatureFusion(
            ch_p3  = self._CH_P3,
            ch_p4  = self._CH_P4,
            ch_p5  = self._CH_P5,
            out_ch = fpn_ch,
        )
        self.head = ClassificationHead(fpn_ch, num_classes, dropout)

        # Initialise only the new (non-pretrained) modules
        self._init_new_weights()

    # ── Weight initialisation (new modules only) ──────────────────────

    def _init_new_weights(self) -> None:
        new_modules = [self.edge_attn, self.scam, self.cbam3d, self.fpn, self.head]
        for module in new_modules:
            for m in module.modules():
                if isinstance(m, (nn.Conv2d, nn.Conv3d)):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm3d, nn.GroupNorm)):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)
                elif isinstance(m, nn.LayerNorm):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)
                elif isinstance(m, nn.Linear):
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        img = x

        # Backbone: strictly sequential, unmodified flow
        f3 = self.f0_3(img)       # raw backbone features at stage 3
        f5 = self.f4_5(f3)        # raw backbone features at stage 5
        f7 = self.f6_7(f5)        # raw backbone features at stage 7

        # FPN P5: apply CBAM3D on f7 then top conv
        p5 = self.f8(self.cbam3d(f7))   # (B, 1536, H/32, W/32) — same as before

        # FPN taps: enrich raw backbone features, NOT the backbone flow
        p3 = self.edge_attn(f3, img)    # side-enrichment of f3
        p4 = self.scam(f5)              # side-enrichment of f5

        return self.head(self.fpn(p3, p4, p5))

    def count_parameters(self) -> Dict[str, int]:
        """Per-subsystem trainable parameter counts for the complexity table."""
        def _n(m: nn.Module) -> int:
            return sum(p.numel() for p in m.parameters() if p.requires_grad)

        return {
            "backbone_f0_3" : _n(self.f0_3),
            "backbone_f4_5" : _n(self.f4_5),
            "backbone_f6_7" : _n(self.f6_7),
            "backbone_f8"   : _n(self.f8),
            "edge_attn"     : _n(self.edge_attn),
            "scam"          : _n(self.scam),
            "cbam3d"        : _n(self.cbam3d),
            "fpn"           : _n(self.fpn),
            "head"          : _n(self.head),
            "total"         : _n(self),
        }


# Load architechture

In [ ]:
def load_architecture(name):

    if name == "stacked_ensembled" or name == "stacked_ensemble_bare":
        return StackedEnsemble(num_classes=NUM_CLASSES)

    elif name == "stacked_ensembled_large":
        return StackedEnsembleLarge(num_classes=NUM_CLASSES)

    elif name == "stacked_ensembled_v2":
        return StackedEnsemble(num_classes=NUM_CLASSES, v2=True)

    elif name == "weighted_ensembled":
        return WeightedEnsemble(num_classes=NUM_CLASSES)

    elif name == "boosted_transformer":
        return BoostedTransformer(num_classes=NUM_CLASSES)

    elif name == "landslide_efficientnet":
        return LandslideEfficientNet(num_classes=NUM_CLASSES)

    # ---------------------
    # RESNET 18
    # ---------------------
    elif name == "resnet18":
        m = models.resnet18(pretrained=False)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
        return m

    # ---------------------
    # MOBILENET V3 Small
    # ---------------------
    elif name == "mobilenet":
        m = models.mobilenet_v3_small(pretrained=False)

        # Replace classifier head safely
        if hasattr(m, "classifier") and len(m.classifier) >= 4:
            m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES)
        else:
            m.classifier = nn.Sequential(
                nn.Linear(m.classifier[0].in_features, NUM_CLASSES)
            )
        return m

    # ---------------------
    # EFFICIENTNET-B3  (TorchVision)
    # ---------------------
    elif name == "efficientnet_b3":
        m = models.efficientnet_b3(pretrained=False)

        # Replace classification layer
        in_features = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
        return m

    elif name == "resnet50":
        m = models.resnet50(pretrained=False)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
        return m

    elif name == "mobilenet_large":
        m = models.mobilenet_v3_large(pretrained=False)

        # Replace classifier head safely
        if hasattr(m, "classifier") and len(m.classifier) >= 4:
            m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES)
        else:
            m.classifier = nn.Sequential(
                nn.Linear(m.classifier[0].in_features, NUM_CLASSES)
            )
        return m

    elif name == "efficientnet_b5":
        m = models.efficientnet_b5(pretrained=False)

        # Replace classification layer
        in_features = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
        return m

    else:
        raise ValueError("Unknown model: %s" % name)


In [ ]:
def load_pretrained_model(model_name, weight_path):
    model = load_architecture(model_name).to(DEVICE)

    if model_name != "landslide_efficientnet":
        model = model.to(memory_format=torch.channels_last)
    state = torch.load(weight_path, map_location=DEVICE)

    for k, v in state.items():
        if "head.proj.weight" in k:
            print("Checkpoint proj weight shape:", v.shape)
            break
    model.load_state_dict(state)
    return model

In [ ]:
def compute_metrics(y_true, y_pred, y_prob, num_classes, class_names=None):
    metrics = {}

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # ---------- Overall ----------
    metrics["overall_accuracy"] = accuracy_score(y_true, y_pred)
    metrics["overall_precision"] = precision_score(y_true, y_pred, average="macro", zero_division=0)
    metrics["overall_recall"] = recall_score(y_true, y_pred, average="macro", zero_division=0)
    metrics["overall_f1"] = f1_score(y_true, y_pred, average="macro", zero_division=0)
    metrics["weighted_f1"] = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    # ---------- Per-class Precision/Recall/F1 ----------
    precisions = precision_score(y_true, y_pred, average=None, zero_division=0)
    recalls = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1s = f1_score(y_true, y_pred, average=None, zero_division=0)

    # ---------- Per-class IoU ----------
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    ious = []
    for cls in range(num_classes):
        TP = cm[cls, cls]
        FP = cm[:, cls].sum() - TP
        FN = cm[cls, :].sum() - TP

        denom = TP + FP + FN
        iou = TP / denom if denom > 0 else 0.0
        ious.append(iou)

    metrics["mean_iou"] = float(np.mean(ious))

    # ---------- Flatten per-class metrics ----------
    for i in range(num_classes):
        cname = class_names[i] if class_names else f"class_{i}"
        cname = cname.replace(" ", "_")

        metrics[f"{cname}_precision"] = float(precisions[i])
        metrics[f"{cname}_recall"]    = float(recalls[i])
        metrics[f"{cname}_f1"]        = float(f1s[i])
        metrics[f"{cname}_iou"]       = float(ious[i])

    # ---------- AUC ----------
    try:
        if num_classes == 2:
            metrics["auc"] = roc_auc_score(y_true, np.array(y_prob)[:, 1])
        else:
            metrics["auc"] = roc_auc_score(y_true, y_prob, multi_class="ovo")
    except:
        metrics["auc"] = None

    metrics["log_loss"] = log_loss(y_true, y_prob)

    return metrics


In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d",
                xticklabels=classes, yticklabels=classes)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
def plot_roc_curve(y_true, y_prob, num_classes):
    y_prob = np.array(y_prob)   # <-- FIX HERE

    plt.figure(figsize=(7, 6))

    if num_classes == 2:
        fpr, tpr, _ = roc_curve(np.array(y_true), y_prob[:, 1])
        plt.plot(fpr, tpr, label="ROC Curve")
    else:
        for c in range(num_classes):
            y_bin = (np.array(y_true) == c).astype(int)
            fpr, tpr, _ = roc_curve(y_bin, y_prob[:, c])
            plt.plot(fpr, tpr, label=f"Class {class_names[c]}")

    plt.plot([0, 1], [0, 1], "k--")
    plt.title("ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()


In [ ]:
from sklearn.model_selection import KFold

def evaluate_model_kfold(model_name, weight_path, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    results = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(dataset)):
        print(f"\n========== Fold {fold+1}/{k} ==========")

        # Test split
        test_subset = Subset(dataset, test_idx)
        test_loader = DataLoader(test_subset, batch_size=32)

        # Load pretrained model
        model = load_pretrained_model(model_name, weight_path)
        model = model.float()
        model.eval()

        y_true, y_pred, y_prob = [], [], []

        with torch.no_grad():
            for x, y in tqdm(test_loader):
                x = x.to(DEVICE, memory_format=torch.channels_last).float()
                y_np = y.cpu().numpy()

                logits = model(x)
                if torch.isnan(logits).any():
                    print("🔥 NaN detected in logits!")
                    raise SystemExit

                probs = torch.softmax(logits, dim=1)
                if torch.isnan(probs).any():
                    print("🔥 NaN detected in probs!")
                    raise SystemExit

                preds = probs.argmax(dim=1).cpu().numpy()
                probs_np = probs.cpu().numpy()

                y_true.extend(y_np)
                y_pred.extend(preds)
                y_prob.extend(probs_np)

        # Compute enhanced metrics
        metrics = compute_metrics(
            y_true, y_pred, y_prob,
            num_classes=NUM_CLASSES,
            class_names=class_names
        )

        print("\n===== METRICS (Fold Results) =====")
        for cname in class_names:
            key = cname.replace(" ", "_")

            print(f"\nClass: {cname}")
            print(f"  Precision: {metrics[f'{key}_precision']:.4f}")
            print(f"  Recall:    {metrics[f'{key}_recall']:.4f}")
            print(f"  F1-score:  {metrics[f'{key}_f1']:.4f}")
            print(f"  IoU:       {metrics[f'{key}_iou']:.4f}")

        print("\nOverall:")
        print(f"  Accuracy:  {metrics['overall_accuracy']:.4f}")
        print(f"  Macro F1:  {metrics['overall_f1']:.4f}")
        print(f"  Mean IoU:  {metrics['mean_iou']:.4f}")

        results.append(metrics)

        # Plots
        plot_confusion_matrix(y_true, y_pred, class_names)
        plot_roc_curve(y_true, y_prob, NUM_CLASSES)

    return results

In [ ]:
compare_results = []

# Stack Ensembled

In [ ]:
model_name = "stacked_ensembled"
weight_path = f"{BASE_MODEL_DIR}/stack_ensembled_best.pth"

results = evaluate_model_kfold(model_name, weight_path, k=5)
compare_results.append((model_name, results))

In [ ]:
df = pd.DataFrame(results)
display(df)

print("\nMean metrics:")
display(df.mean())

# Resnet18 Evaluation

In [ ]:
model_name = "resnet18"
weight_path = f"{BASE_MODEL_DIR}/resnet18_best.pth"

results = evaluate_model_kfold(model_name, weight_path, k=5)
compare_results.append((model_name, results))

In [ ]:
df = pd.DataFrame(results)
display(df)

print("\nMean metrics:")
display(df.mean())

# Mobilenet Evaluation

In [ ]:
model_name = "mobilenet"
weight_path = f"{BASE_MODEL_DIR}/mobilenet_best.pth"

results = evaluate_model_kfold(model_name, weight_path, k=5)
compare_results.append((model_name, results))

In [ ]:
df = pd.DataFrame(results)
display(df)

print("\nMean metrics:")
display(df.mean())

# Efficientnet_b3 Evaluation

In [ ]:
model_name = "efficientnet_b3"
weight_path = f"{BASE_MODEL_DIR}/efficientnet_b3_best.pth"

results = evaluate_model_kfold(model_name, weight_path, k=5)
compare_results.append((model_name, results))

In [ ]:
df = pd.DataFrame(results)
display(df)

print("\nMean metrics:")
display(df.mean())

# Comparing results

In [ ]:
METRIC_COLUMNS = [
    "overall_accuracy",
    "overall_precision",
    "overall_recall",
    "overall_f1",
    "weighted_f1",
    "mean_iou",
    "non_landslide_precision",
    "non_landslide_recall",
    "non_landslide_f1",
    "non_landslide_iou",
    "low_risk_precision",
    "low_risk_recall",
    "low_risk_f1",
    "low_risk_iou",
    "high_risk_precision",
    "high_risk_recall",
    "high_risk_f1",
    "high_risk_iou",
    "auc",
    "log_loss"
]


In [ ]:
import pandas as pd
import numpy as np

def build_comparison_table(compare_results, metric_columns):
    rows = []

    for model_name, results in compare_results:
        # Normalize results to list of fold dicts
        if isinstance(results, dict):
            fold_results = list(results.values())
        else:
            fold_results = results  # already a list

        model_row = {"model": model_name}

        for metric in metric_columns:
            values = [
                fold[metric]
                for fold in fold_results
                if metric in fold and fold[metric] is not None
            ]

            model_row[metric] = np.mean(values) if values else np.nan

        rows.append(model_row)

    df = pd.DataFrame(rows)
    return df


In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
comparison_df = build_comparison_table(compare_results, METRIC_COLUMNS)

comparison_df.sort_values(by="weighted_f1", ascending=False, inplace=True)
comparison_df

In [ ]:
comparison_df.T